In [2]:
import os
import json
import gzip
import pandas as pd
import numpy as np

import scipy.sparse as spsparse
import scipy.stats as spstats


from statsmodels.stats.multitest import multipletests as holm
import statsmodels.api as sm

import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
import matplotlib.transforms as transforms

import seaborn as sns


# Load the data

In [2]:
def data_preprocess(country_auc_df):
    
    country_auc_df['zscore'] = (country_auc_df['AUC'] - 0.5)/np.sqrt(country_auc_df['Cov'])
    country_auc_df['pvalue'] = spstats.norm.sf(np.abs(country_auc_df['zscore'].values))
    
    country_auc_df['significant'] = country_auc_df['pvalue']<=0.01
    
    self_auc = country_auc_df[country_auc_df['CitingCountry'] == country_auc_df['CitedCountry']].sort_values(by=['CitedYear', 'CitedCountry']).reset_index(drop=True)
        
    self_auc.rename(columns={'CitedYear':'Year', 'CitedCountry':'Country'}, inplace=True)
    
    return self_auc
    
#from scipy.stats import zscore

#name=None
name='noselfauthor'

if name=='noselfauthor':
    country_auc_df=pd.read_csv('CountryData2/oa_countrycites_noselfauthor_auc.csv.gz')
    
elif name=='noselfauthoraff':
    country_auc_df=pd.read_csv('CountryData2/oa_countrycites_noselfauthoraff_auc.csv.gz')
    
else:
    print('------ \n No file request \n ------- ')
    country_auc_df=pd.DataFrame()

filename_suffix=name 

if not country_auc_df.empty:

    self_auc=data_preprocess(country_auc_df)

In [4]:
self_auc

,CitingCountry,Year,Country,AUC,Cov,N,zscore,pvalue,significant
0,GB,1848,GB,0.500000,NaN,1,NaN,NaN,False
1,DE,1858,DE,0.500000,NaN,1,NaN,NaN,False
2,DE,1860,DE,0.500000,NaN,1,NaN,NaN,False
3,DE,1863,DE,0.500000,NaN,1,NaN,NaN,False
4,DE,1864,DE,0.500000,NaN,1,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...
10987,XK,2024,XK,0.497925,NaN,1,NaN,NaN,False
10988,YE,2024,YE,0.489022,0.000005,15,-4.740404,0.000001,True
10989,ZA,2024,ZA,0.530997,0.000045,558,4.640876,0.000002,True
10990,ZM,2024,ZM,0.572781,0.001809,22,1.711071,0.043534,False


# Bootstrap

In [354]:
def data_preprocess_bootstrap(country_auc_df):

    country_auc_df = country_auc_df[country_auc_df['N']>=2].reset_index(drop=True)
    
    country_auc_df['STD'] = country_auc_df.groupby(['CitingCountry', 'CitedYear', 'CitedCountry'])['AUC'].transform('std')
    country_auc_df['AUC'] = country_auc_df.groupby(['CitingCountry', 'CitedYear', 'CitedCountry'])['AUC'].transform('mean')
    
    country_auc_df.drop_duplicates(['CitingCountry', 'CitedYear', 'CitedCountry'], inplace=True)
    del country_auc_df['isample']
    
    country_auc_df['zscore'] = (country_auc_df['AUC'] - 0.5)/country_auc_df['STD']
    country_auc_df['pvalue'] = spstats.norm.sf(np.abs(country_auc_df['zscore'].values))
    
    def reject_holm(pvalues):
        reject, pvalues, _, _ = holm(pvalues, alpha=0.01, method='hs', is_sorted=False, returnsorted=False)
        return reject
    country_auc_df['significant'] = country_auc_df.groupby('CitedYear')['pvalue'].transform(reject_holm)
    
    self_auc = country_auc_df[country_auc_df['CitingCountry'] == country_auc_df['CitedCountry']].sort_values(by=['CitedYear', 'CitedCountry']).reset_index(drop=True)
    self_auc.rename(columns={'CitedYear':'Year', 'CitedCountry':'Country'}, inplace=True)


    return self_auc

In [6]:
name = None
#name='noselfauthoraff_aucboot'

if name =='noselfauthor_aucboot':
   country_auc_bootstrap = pd.read_csv(os.path.join('CountryData2', 'oa_countrycites_noselfauthor_aucboot.csv.gz'))
   filename_suffix='bootstrap_noselfauthor'

elif name =='noselfauthoraff_aucboot':
   country_auc_bootstrap = pd.read_csv(os.path.join('CountryData2', 'oa_countrycites_noselfauthoraff_aucboot.csv.gz'))
   filename_suffix='bootstrap_noselfauthoraff'

else:
    print('------ \n No file request \n ------- ')
    country_auc_bootstrap=pd.DataFrame()

if not country_auc_bootstrap.empty:
    self_auc = data_preprocess_bootstrap(country_auc_bootstrap)

------ 
 No file request 
 ------- 


# Statistics

In [3]:
print('---------- basic information: start year, end year, number of countries')

self_auc['Year'].min(), self_auc['Year'].max(), self_auc['Country'].nunique()

---------- basic information: start year, end year, number of countries


(1848, 2024, 221)

In [8]:
self_auc[self_auc['Year']==2024]

,CitingCountry,Year,Country,AUC,Cov,N,zscore,pvalue,significant
10842,AE,2024,AE,0.523044,0.000187,121,1.686765,0.045824,False
10843,AF,2024,AF,0.498368,0.001284,15,-0.045560,0.481831,False
10844,AG,2024,AG,0.500000,NaN,1,NaN,NaN,False
10845,AL,2024,AL,0.602494,0.003763,13,1.670890,0.047372,False
10846,AM,2024,AM,0.492840,0.000008,12,-2.464273,0.006865,True
...,...,...,...,...,...,...,...,...,...
10987,XK,2024,XK,0.497925,NaN,1,NaN,NaN,False
10988,YE,2024,YE,0.489022,0.000005,15,-4.740404,0.000001,True
10989,ZA,2024,ZA,0.530997,0.000045,558,4.640876,0.000002,True
10990,ZM,2024,ZM,0.572781,0.001809,22,1.711071,0.043534,False


# other variables

In [4]:
countryprodnorm = pd.read_csv('CountryData2/oa_countrynormtopproductivity.csv.gz')
countryprodnorm 

,Year,Country,NumPub,TopJournal,FracTop,OANumPub,OATopJournal,OAFracTop,NormTopFrac
0,11.0,US,1,0.0,0.0,1,0.0,0.0,NaN
1,21.0,TR,5,0.0,0.0,5,0.0,0.0,NaN
2,22.0,TR,7,0.0,0.0,7,0.0,0.0,NaN
3,22.0,US,1,0.0,0.0,7,0.0,0.0,NaN
4,204.0,TR,1,0.0,0.0,1,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...
22803,2025.0,JP,1,0.0,0.0,28,0.0,0.0,NaN
22804,2025.0,KR,1,0.0,0.0,28,0.0,0.0,NaN
22805,2025.0,PK,1,0.0,0.0,28,0.0,0.0,NaN
22806,2025.0,TH,2,0.0,0.0,28,0.0,0.0,NaN


In [9]:
def compile_dataset(self_auc):

    # number of publications and top journal pubs, and the fraction of top journal pubs
    # countryprod = pd.read_csv('CountryData2/oa_countrytopproductivity.csv.gz')
    # countryprod['TopJournal']=countryprod['TopJournal'].replace({'True':'1'})
    # countryprod['TopJournal']=countryprod['TopJournal'].replace({'False':'0'})
    # countryprod['TopJournal']=countryprod['TopJournal'].astype(float)


    # worldwide_frac_top = countryprod.groupby('Year').agg({
    #     'TopJournal': 'sum',
    #     'NumPub': 'sum'
    # }).reset_index()
    
    # worldwide_frac_top['worldwide_frac_top'] = worldwide_frac_top['TopJournal'] / worldwide_frac_top['NumPub']
    
    # countryprod = pd.merge(countryprod, worldwide_frac_top[['Year', 'worldwide_frac_top']], on='Year')
    
    # countryprod['normalized_frac_top'] = countryprod['FracTop'] / countryprod['worldwide_frac_top']
    countryprodnorm = pd.read_csv('CountryData2/oa_countrynormtopproductivity.csv.gz')
    countryprodnorm.rename(columns={'NormTopFrac':'normalized_frac_top'}, inplace=True)
    
    countryprodnorm['logNumPub'] = np.log10(countryprodnorm['NumPub']+1)
    self_auc = self_auc.merge(countryprodnorm, how='left', on=['Country', 'Year'])
    
    #oasize = pd.read_csv('CountryData/globaltopjournal.csv')
    #self_auc = self_auc.merge(oasize, how='left', on='Year')
    #patents = pd.read_csv('/Users/hgt6rn/Downloads/resulttable.csv', sep=';')
    #patents = patents.rename(columns={'appln_filing_year':'Year', 'person_ctry_code':'Country'})
    
    #self_auc = self_auc.merge(patents, how='left', on=['Year', 'Country'])
    
    #self_auc['NormedTop'] = self_auc['FracTop'].values/self_auc['OAFracTop'].values
    
    national_authorship_rate = pd.read_csv( "CountryData2/oa_national_authorship_rate.csv.gz")
    
    national_authorship_rate['FracInternationalAuthors']=1-national_authorship_rate['FractionNationalAuthors']
    self_auc = self_auc.merge(national_authorship_rate, how='left', on=['Country', 'Year'])
    
    pubtopics1_div = pd.read_csv("CountryData/oa_country_topic1_diversity.csv.gz")
    self_auc = self_auc.merge(pubtopics1_div, how='left', on=['Country', 'Year'])
    self_auc.rename(columns={'TopicDiversity':'TopicDiversity1'}, inplace=True)
    self_auc['TopicDiversity1'] = 1 - self_auc['TopicDiversity1']
    
    pubtopics2_div = pd.read_csv("CountryData/oa_country_topic2_diversity.csv.gz")
    self_auc = self_auc.merge(pubtopics2_div, how='left', on=['Country', 'Year'])
    self_auc.rename(columns={'TopicDiversity':'TopicDiversity2'}, inplace=True)
    self_auc['TopicDiversity2'] = 1 - self_auc['TopicDiversity2']
    
    wbgdp=pd.read_csv('CountryData/worldbank_indicators.csv')
    del wbgdp['country_name']
    wbgdp.rename(columns={'country_code':'Country', 'year':'Year',
                          'Research and development expenditure (% of GDP)':'RND_per',
                      'Patent applications, nonresidents':'PAT_res', 'Patent applications, residents': 'PAT_nres',
                      'GDP (constant 2015 US$)':'GDP','GDP per capita (constant 2015 US$)':'GDP_PCAP',
                       'GNI (constant 2015 US$)': 'GNI','GNI per capita (constant 2015 US$)':'GNI_PCAP',
                      'Researchers in R&D (per million people)':'NResearchers',
                      'Population, Total':'Pop'}, inplace=True)
    
    #wbgdp = pd.read_csv('CountryData/world-bank-data.csv')
    #wbgdp['frac_researchers'] = wbgdp.groupby('Year')['NResearchers'].transform(lambda x: x / x.sum())
    
    wbgdp['PAT_total'] = wbgdp['PAT_nres'] + wbgdp['PAT_res']
    for c in ['GDP', 'GNI', 'GDP_PCAP', 'GNI_PCAP', 'PAT_nres', 'PAT_res', 'PAT_total', ]:   #'GNI_PCAP', 'GDP_PCAP', 'RND_per'
        wbgdp[c] = np.log10(wbgdp[c].astype(float))
        
    wbgdp['logPop']= np.log10(wbgdp['Pop'])
    
    self_auc = self_auc.merge(wbgdp, how='left', on=['Country', 'Year'])

    return self_auc

In [10]:
dfself_vars=compile_dataset(self_auc)


In [374]:
dfself_vars=compile_dataset(self_auc)

dfself_vars.sort_values(by=['Country','Year'],inplace=True)
dfself_vars['zscore'] = dfself_vars['zscore'].replace(-np.inf, 0)
#dfself_vars['logzscore']=np.log10(dfself_vars['zscore']-dfself_vars['zscore'].min()+1)
#dfself_vars['logzscore_bootstrap']=np.log10(dfself_vars['zscore_bootstrap'])

dfself_vars['pub_capita']=dfself_vars['NumPub']/dfself_vars['Pop']
# dfself_vars['citations_per_pub']=dfself_vars['AllCitations']/dfself_vars['NumPub']
# no data of allcitations for recent years

def income():
    
    income=pd.read_excel('CountryData/OGHIST_WorldBank.xlsx', sheet_name=2, skiprows=10)
    
    new_column_names = ['iso3','country']+list(range(1987, 2024))
    income.columns = new_column_names
    income['iso3']=income['iso3'].str.strip()
    del income['country']
    
    country_codes=pd.read_csv('CountryData/Gravity_csv_V202211/Countries_V202211.csv')
    country_codes['iso2']=country_codes['iso2'].str.strip()
    country_codes['iso3']=country_codes['iso3'].str.strip()
    country_dict = dict(zip(country_codes['iso3'], country_codes['iso2']))
    income['iso3'] = income['iso3'].map(country_dict)
    
    income=income[income['iso3'].notnull()]
    long_data = pd.melt(income, id_vars=["iso3"], var_name="Year", value_name="income_level")
    long_data.rename(columns={'iso3':'Country'}, inplace=True)

    return long_data

income=income()
    
dfself_vars=dfself_vars.merge(income, how='left', on=['Year','Country'])


dfself_vars.to_csv(f'RegressionSelfCitation/SelfCitation_Plus_Covariates_{filename_suffix}_07172025.csv', index=False, sep=',')

In [375]:
filename_suffix

'bootstrap_noselfauthoraff'

In [376]:
dfself_vars.columns

Index(['CitingCountry', 'Year', 'Country', 'AUC', 'N', 'STD', 'zscore',
       'pvalue', 'significant', 'NumPub', 'TopJournal', 'FracTop', 'OANumPub',
       'OATopJournal', 'OAFracTop', 'normalized_frac_top', 'logNumPub',
       'FractionNationalAuthors', 'FracInternationalAuthors',
       'TopicDiversity1', 'TopicDiversity2', 'RND_per', 'PAT_res', 'PAT_nres',
       'GDP', 'GDP_PCAP', 'GNI', 'GNI_PCAP', 'NResearchers', 'Pop',
       'PAT_total', 'logPop', 'pub_capita', 'income_level'],
      dtype='object')

In [1]:
import pandas as pd

In [2]:
income=pd.read_excel('CountryData/OGHIST_WorldBank.xlsx', sheet_name=2, skiprows=10)


In [4]:
country_codes=pd.read_csv('CountryData/Gravity_csv_V202211/Countries_V202211.csv')


In [5]:
country_codes

,country_id,iso3,iso3num,country,countrylong,first_year,last_year,countrygroup_iso3,countrygroup_iso3num,iso2,heg_iso3_2020,heg_iso3num_2020
0,AFG,AFG,4.0,Afghanistan,Islamic Republic of Afghanistan,NaN,NaN,NaN,NaN,AF,NaN,NaN
1,ALB,ALB,8.0,Albania,Republic of Albania,NaN,NaN,NaN,NaN,AL,NaN,NaN
2,DZA,DZA,12.0,Algeria,People's Democratic Republic of Algeria,NaN,NaN,NaN,NaN,DZ,NaN,NaN
3,ASM,ASM,16.0,American Samoa,American Samoa,NaN,NaN,NaN,NaN,AS,USA,840.0
4,AND,AND,20.0,Andorra,Principality of Andorra,NaN,NaN,NaN,NaN,AD,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
247,SDN.2,SDN,729.0,Sudan,Republic of the Sudan,2011.0,NaN,SDN,736.0,SD,NaN,NaN
248,IDN.2,IDN,360.0,Indonesia,Republic of Indonesia,2002.0,NaN,IDN,360.0,ID,NaN,NaN
249,DEU.2,DEU,276.0,Germany,Federal Republic of Germany,1990.0,NaN,DEU,276.0,DE,NaN,NaN
250,YEM.2,YEM,887.0,Yemen,Republic of Yemen,1990.0,NaN,YEM,887.0,YE,NaN,NaN


In [6]:
income

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38
0,AFG,Afghanistan,L,L,L,L,L,L,L,L,...,L,L,L,L,L,L,L,L,L,L
1,ALB,Albania,..,..,..,LM,LM,LM,L,L,...,UM,UM,UM,UM,UM,UM,UM,UM,UM,UM
2,DZA,Algeria,UM,UM,LM,LM,LM,LM,LM,LM,...,UM,UM,UM,UM,UM,LM,LM,LM,LM,UM
3,ASM,American Samoa,H,H,H,UM,UM,UM,UM,UM,...,UM,UM,UM,UM,UM,UM,UM,UM,H,H
4,AND,Andorra,..,..,..,H,H,H,H,H,...,H,H,H,H,H,H,H,H,H,H
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224,YUG,Serbia and Montenegro (former),..,..,..,..,..,LM,LM,LM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
225,SUN,USSR (former),..,..,..,UM,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
226,YUGf,Yugoslavia (former),UM,UM,UM,UM,UM,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
